# 06. 전처리 및 파생변수 생성

**입력**: `data/raw/` 의 6개 CSV 파일  
**출력**: `data/processed/` 의 5개 CSV 파일

| 출력 파일 | 설명 | 크기 |
|-----------|------|------|
| `df_usdt.csv` | USDT 전용 데이터셋 | 2,302행 × 88열 |
| `df_usdc.csv` | USDC 전용 데이터셋 | 2,302행 × 86열 |
| `df_dai.csv`  | DAI 전용 데이터셋 | 2,302행 × 87열 |
| `df_fiat.csv` | USDT+USDC 통합 (법정담보 모델용) | 4,604행 × 86열 |
| `df_merged.csv` | 전체 wide format (통합 모델용) | 2,316행 × 227열 |

## 처리 단계
1. 각 데이터 로딩 및 개별 전처리 (결측치 처리)
2. Date 기준 전체 병합
3. 타깃 변수 생성 (동적 임계값 기반 디페깅 레이블)
4. 파생변수 생성 (~40~50개/코인)
5. 코인별 데이터셋 분리 및 저장
6. 실제 데이터 확인

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR  = os.path.join('..', 'data', 'raw')
OUT_DIR  = os.path.join('..', 'data', 'processed')
os.makedirs(OUT_DIR, exist_ok=True)

START_DATE = '2019-11-22'

---
## Step 1. 데이터 로딩 및 결측치 처리

In [2]:
def load_cmc():
    """
    CMC 데이터 로딩
    - 100% 결측 컬럼 제거: circulating_supply, total_supply, max_supply
      → CoinMarketCap API가 현재 시점 공급량만 반환 (역사 일별 X)
      → 대신 onchain_supply.csv의 circ_* 컬럼 사용
    """
    df = pd.read_csv(os.path.join(RAW_DIR, 'cmc_market_info.csv'))
    df['Date'] = pd.to_datetime(df['Date'])
    drop_cols = [c for c in df.columns if df[c].isnull().mean() == 1.0]
    df = df.drop(columns=drop_cols)
    keep = ['Date'] + [c for c in df.columns if any(
        c.startswith(p) for p in ['open_', 'high_', 'low_', 'close_', 'volume_', 'market_cap_']
    )]
    df = df[keep].sort_values('Date').reset_index(drop=True)
    print(f'CMC: {df.shape}')
    return df


def load_macro():
    """
    거시경제 데이터 결측치 처리
    - DXY, VIX: 주말/공휴일에 시장 닫힘 → 직전 영업일 값으로 채움 (ffill)
    - federal_funds_rate: 월 1회 발표 → 발표일 이후 다음 발표까지 동일 값 유지 (ffill)
    """
    df = pd.read_csv(os.path.join(RAW_DIR, 'macro_data.csv'))
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['dxy'] = df['dxy'].ffill()
    df['vix'] = df['vix'].ffill()
    df['federal_funds_rate'] = df['federal_funds_rate'].ffill().fillna(0)
    print(f'Macro: {df.shape} | 잔여 결측: {df.isnull().sum().sum()}')
    return df


def load_fgi():
    """
    공포탐욕지수 로딩
    - timestamp 컬럼 → Date로 변환
    - value: 0(극단적 공포) ~ 100(극단적 탐욕)
    """
    df = pd.read_csv(os.path.join(RAW_DIR, 'fear_and_greed_index.csv'))
    df['Date'] = pd.to_datetime(df['timestamp'])
    df = df[['Date', 'value', 'value_classification']].copy()
    df.columns = ['Date', 'fgi', 'fgi_class']
    df = df.sort_values('Date').reset_index(drop=True)
    print(f'FGI: {df.shape}')
    return df


def load_onchain():
    """
    온체인 공급량 데이터 결측치 처리
    - minted_* 컬럼: 100% 결측 → 제거
    - supply_change 첫 행: diff() 계산 시 발생하는 NaN → 0 처리
    - unreleased_USDC (49% 결측): ffill 후 0으로 채움
    """
    df = pd.read_csv(os.path.join(RAW_DIR, 'onchain_supply.csv'))
    df['Date'] = pd.to_datetime(df['Date'])
    drop_cols = [c for c in df.columns if df[c].isnull().mean() == 1.0]
    df = df.drop(columns=drop_cols)
    change_cols = [c for c in df.columns if 'supply_change' in c]
    df[change_cols] = df[change_cols].fillna(0)
    df['unreleased_USDC'] = df['unreleased_USDC'].ffill().fillna(0)
    df['unreleased_USDT'] = df['unreleased_USDT'].ffill().fillna(0)
    df = df.sort_values('Date').reset_index(drop=True)
    print(f'Onchain: {df.shape}')
    return df


print('=== 데이터 로딩 ===')
cmc     = load_cmc()
macro   = load_macro()
fgi     = load_fgi()
onchain = load_onchain()

=== 데이터 로딩 ===
CMC: (2317, 31)
Macro: (2317, 5) | 잔여 결측: 1
FGI: (2971, 3)
Onchain: (2317, 24)


---
## Step 2. 전체 병합

In [3]:
# Date 기준으로 4개 데이터 병합 (left join: CMC 기준)
df = cmc.copy()
df = df.merge(macro,   on='Date', how='left')
df = df.merge(fgi,     on='Date', how='left')
df = df.merge(onchain, on='Date', how='left')

df = df[df['Date'] >= START_DATE].sort_values('Date').reset_index(drop=True)

# FGI: 날짜 불일치로 발생하는 결측 → ffill/bfill
df['fgi']       = df['fgi'].ffill().bfill()
df['fgi_class'] = df['fgi_class'].ffill().bfill()

print(f'병합 완료: {df.shape}')
print(f'기간: {df["Date"].min().date()} ~ {df["Date"].max().date()}')

병합 완료: (2316, 60)
기간: 2019-11-22 ~ 2026-03-25


---
## Step 3. 타깃 변수 생성 (동적 임계값)

**공식**: Thresh = 1 ± 10 / V_monthly^(1/3)

- V_monthly: 해당 월의 평균 일별 거래량
- 거래량이 많을수록 임계값이 좁아짐
- **디페깅 레이블**: `close < Thresh_low` OR `close > Thresh_high` → Y=1

In [4]:
def make_target(df, coin):
    vol_col   = f'volume_{coin}'
    close_col = f'close_{coin}'
    month = df['Date'].dt.to_period('M')

    # Kaiko 원본 방식: 이전 달 거래량으로 임계값 계산 (data leakage 방지)
    # 3월 2일 → 2월 전체 거래량 사용 (3월이 끝나기 전엔 3월 거래량 알 수 없음)
    monthly_vol      = df.groupby(month)[vol_col].mean()
    prev_monthly_vol = monthly_vol.shift(1)  # 이전 달로 밀기
    df[f'V_monthly_{coin}'] = month.map(prev_monthly_vol)

    df[f'thresh_low_{coin}']  = 1 - 10 / (df[f'V_monthly_{coin}'] ** (1/3))
    df[f'thresh_high_{coin}'] = 1 + 10 / (df[f'V_monthly_{coin}'] ** (1/3))
    df[f'depeg_{coin}'] = (
        (df[close_col] < df[f'thresh_low_{coin}']) |
        (df[close_col] > df[f'thresh_high_{coin}'])
    ).astype(int)

    n       = df[f'depeg_{coin}'].sum()
    nan_cnt = df[f'V_monthly_{coin}'].isna().sum()
    tl      = df[f'thresh_low_{coin}'].dropna().iloc[0]
    th      = df[f'thresh_high_{coin}'].dropna().iloc[0]
    print(f'{coin}: depeg {n}/{len(df)} ({n/len(df)*100:.1f}%) | 첫달 NaN={nan_cnt}행 | 임계값 예시 {tl:.4f}~{th:.4f}')
    return df


print('=== 타깃 변수 생성 ===')
for coin in ['USDT', 'USDC', 'DAI']:
    df = make_target(df, coin)


=== 타깃 변수 생성 ===
USDT: depeg 113/2316 (4.9%) | 첫달 NaN=9행 | 임계값 예시 0.9966~1.0034
USDC: depeg 28/2316 (1.2%) | 첫달 NaN=9행 | 임계값 예시 0.9846~1.0154
DAI: depeg 13/2316 (0.6%) | 첫달 NaN=9행 | 임계값 예시 0.8970~1.1030


---
## Step 4. 파생변수 생성

In [5]:
def make_features(df, coin):
    """코인별 가격/거래량/시가총액 파생변수 생성"""
    p = f'close_{coin}'
    v = f'volume_{coin}'
    m = f'market_cap_{coin}'
    # 수익률
    df[f'return_1d_{coin}'] = df[p].pct_change(1)
    df[f'return_3d_{coin}'] = df[p].pct_change(3)
    df[f'return_7d_{coin}'] = df[p].pct_change(7)
    # 디페깅 크기
    df[f'depeg_magnitude_{coin}'] = (df[p] - 1.0).abs()
    # 캔들 패턴
    df[f'high_low_spread_{coin}'] = (df[f'high_{coin}'] - df[f'low_{coin}']) / df[p]
    df[f'upper_shadow_{coin}']    = (df[f'high_{coin}'] - df[p]) / df[p]
    df[f'lower_shadow_{coin}']    = (df[p] - df[f'low_{coin}']) / df[p]
    # 이동평균 이탈
    df[f'ma7_{coin}']      = df[p].rolling(7).mean()
    df[f'ma30_{coin}']     = df[p].rolling(30).mean()
    df[f'ma7_dev_{coin}']  = df[p] - df[f'ma7_{coin}']
    df[f'ma30_dev_{coin}'] = df[p] - df[f'ma30_{coin}']
    # 변동성
    df[f'vol_7d_{coin}']  = df[f'return_1d_{coin}'].rolling(7).std()
    df[f'vol_30d_{coin}'] = df[f'return_1d_{coin}'].rolling(30).std()
    # RSI(14)
    delta = df[p].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    df[f'rsi_14_{coin}'] = 100 - (100 / (1 + rs))
    # 거래량
    df[f'vol_ma7_{coin}']      = df[v].rolling(7).mean()
    df[f'volume_ratio_{coin}'] = df[v] / df[f'vol_ma7_{coin}'].replace(0, np.nan)
    df[f'volume_surge_{coin}'] = (df[f'volume_ratio_{coin}'] > 2).astype(int)
    df[f'turnover_rate_{coin}']= df[v] / df[m].replace(0, np.nan)
    # 시가총액 변화율
    df[f'mcap_change_1d_{coin}'] = df[m].pct_change(1)
    df[f'mcap_change_7d_{coin}'] = df[m].pct_change(7)
    return df


def make_onchain_features(df, coin):
    """온체인 공급량 파생변수 (Exchange Flow 대리변수)"""
    sc   = f'supply_change_{coin}'
    circ = f'circ_{coin}'
    m    = f'market_cap_{coin}'
    if sc in df.columns and circ in df.columns:
        df[f'mint_intensity_{coin}']     = df[sc] / df[m].replace(0, np.nan)
        df[f'supply_growth_rate_{coin}'] = df[sc] / df[circ].shift(1).replace(0, np.nan)
    return df


def make_macro_features(df):
    """거시경제 파생변수 및 복합 위험 신호"""
    df['dxy_change_1d']  = df['dxy'].pct_change(1)
    df['dxy_change_7d']  = df['dxy'].pct_change(7)
    df['vix_change_1d']  = df['vix'].pct_change(1)
    df['vix_spike']      = (df['vix'] > 30).astype(int)
    df['rate_hike']      = (df['federal_funds_rate'].diff() > 0).astype(int)
    df['btc_dom_change'] = df['btc_dominance'].pct_change(7)
    df['risk_off']       = ((df['vix'] > 25) & (df['dxy_change_7d'] > 0)).astype(int)
    df['btc_return_1d']  = df['close_BTC'].pct_change(1)
    df['eth_return_1d']  = df['close_ETH'].pct_change(1)
    df['btc_crash']      = (df['btc_return_1d'] < -0.05).astype(int)
    df['btc_vol_7d']     = df['btc_return_1d'].rolling(7).std()
    df['eth_vol_7d']     = df['eth_return_1d'].rolling(7).std()
    df['crypto_stress']  = ((df['btc_return_1d'] < -0.10) & (df['vix'] > 30)).astype(int)
    df['fgi_change_1d']  = df['fgi'].diff()
    df['extreme_fear']   = (df['fgi'] < 20).astype(int)
    df['extreme_greed']  = (df['fgi'] > 80).astype(int)
    return df


def make_cross_coin_features(df):
    """코인 간 관계 변수"""
    df['usdt_usdc_spread']      = df['close_USDT'] - df['close_USDC']
    df['usdt_dai_spread']       = df['close_USDT'] - df['close_DAI']
    total_vol = df['volume_USDT'] + df['volume_USDC'] + df['volume_DAI']
    df['vol_share_USDT']        = df['volume_USDT'] / total_vol.replace(0, np.nan)
    df['vol_share_USDC']        = df['volume_USDC'] / total_vol.replace(0, np.nan)
    df['vol_share_DAI']         = df['volume_DAI']  / total_vol.replace(0, np.nan)
    df['usdt_usdc_mcap_ratio']  = df['market_cap_USDT'] / df['market_cap_USDC'].replace(0, np.nan)
    df['dai_eth_return_corr_30d'] = df['return_1d_DAI'].rolling(30).corr(df['eth_return_1d'])
    return df


def make_lag_features(df, coin, lag_cols_base):
    """라그 변수 생성 (1/3/7일 전 값)"""
    for col in lag_cols_base:
        full_col = f'{col}_{coin}' if f'{col}_{coin}' in df.columns else col
        if full_col not in df.columns:
            continue
        for lag in [1, 3, 7]:
            df[f'{full_col}_lag{lag}'] = df[full_col].shift(lag)
    return df


print('=== 파생변수 생성 ===')
df = df.copy()
for coin in ['USDT', 'USDC', 'DAI']:
    df = make_features(df, coin)
    df = make_onchain_features(df, coin)
    print(f'  {coin} 완료')
df = make_macro_features(df)
df = make_cross_coin_features(df)
print('  거시경제 / 코인 간 변수 완료')
LAG_BASE = ['return_1d', 'vol_7d', 'volume_ratio', 'depeg_magnitude', 'supply_change', 'mcap_change_1d']
for coin in ['USDT', 'USDC', 'DAI']:
    df = make_lag_features(df, coin, LAG_BASE)
df = make_lag_features(df, '', ['btc_return_1d', 'eth_return_1d', 'vix', 'dxy'])
print(f'  라그 변수 완료 | 총 변수 수: {df.shape[1]}')

=== 파생변수 생성 ===
  USDT 완료
  USDC 완료
  DAI 완료
  거시경제 / 코인 간 변수 완료
  라그 변수 완료 | 총 변수 수: 227


C:\Users\yusse\AppData\Local\Temp\ipykernel_19680\1793067807.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{full_col}_lag{lag}'] = df[full_col].shift(lag)
C:\Users\yusse\AppData\Local\Temp\ipykernel_19680\1793067807.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{full_col}_lag{lag}'] = df[full_col].shift(lag)
C:\Users\yusse\AppData\Local\Temp\ipykernel_19680\1793067807.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

---
## Step 5. 코인별 데이터셋 분리 및 저장

In [6]:
def build_coin_df(df, coin):
    """특정 코인에 필요한 컬럼만 추출"""
    coin_cols   = [c for c in df.columns if coin in c]
    common_cols = [
        'Date', 'dxy', 'vix', 'federal_funds_rate', 'btc_dominance',
        'dxy_change_1d', 'dxy_change_7d', 'vix_change_1d',
        'vix_spike', 'rate_hike', 'btc_dom_change', 'risk_off',
        'btc_return_1d', 'eth_return_1d', 'btc_crash',
        'btc_vol_7d', 'eth_vol_7d', 'crypto_stress',
        'fgi', 'fgi_class', 'fgi_change_1d', 'extreme_fear', 'extreme_greed',
        'usdt_usdc_spread', 'usdt_dai_spread',
        'vol_share_USDT', 'vol_share_USDC', 'vol_share_DAI',
        'usdt_usdc_mcap_ratio',
    ]
    if coin == 'DAI':
        common_cols += ['dai_eth_return_corr_30d']
    all_cols = [c for c in common_cols + coin_cols if c in df.columns]
    all_cols = list(dict.fromkeys(all_cols))
    return df[all_cols].copy()


# minted_* 컬럼 제거: onchain raw 데이터에서 100% NaN으로 들어온 컬럼
# DefiLlama API가 totalMintedUSD를 항상 채워주지 않아 전처리 후에도 전부 결측
minted_cols = [c for c in df.columns if c.startswith('minted_') or c.startswith('eth_minted_') or c.startswith('unreleased_')]
if minted_cols:
    df = df.drop(columns=minted_cols)
    print(f'전부 NaN 컬럼 {len(minted_cols)}개 제거: {minted_cols}')

# dropna 기준: ma30(30일 rolling)이 가장 늦게 채워지는 컬럼이므로 이 기준으로 초기 결측 제거
# rsi_14 기준(14행 제거)으로는 ma30/vol_30d(30행 필요) NaN이 잔존
df_dai  = build_coin_df(df, 'DAI').dropna(subset=['ma30_DAI']).reset_index(drop=True)
df_usdt = build_coin_df(df, 'USDT').dropna(subset=['ma30_USDT']).reset_index(drop=True)
df_usdc = build_coin_df(df, 'USDC').dropna(subset=['ma30_USDC']).reset_index(drop=True)

df_dai.to_csv(os.path.join(OUT_DIR, 'df_dai.csv'),   index=False)
df_usdt.to_csv(os.path.join(OUT_DIR, 'df_usdt.csv'), index=False)
df_usdc.to_csv(os.path.join(OUT_DIR, 'df_usdc.csv'), index=False)

# 법정담보 통합 (USDT + USDC long format)
df_usdt_l = df_usdt.copy(); df_usdt_l['coin'] = 'USDT'
df_usdc_l = df_usdc.copy(); df_usdc_l['coin'] = 'USDC'
df_usdt_l = df_usdt_l.rename(columns={c: c.replace('_USDT', '_coin') for c in df_usdt_l.columns})
df_usdc_l = df_usdc_l.rename(columns={c: c.replace('_USDC', '_coin') for c in df_usdc_l.columns})
common = list(set(df_usdt_l.columns) & set(df_usdc_l.columns))
df_fiat = pd.concat([df_usdt_l[common], df_usdc_l[common]], ignore_index=True)
df_fiat = df_fiat.sort_values(['Date', 'coin']).reset_index(drop=True)
df_fiat.to_csv(os.path.join(OUT_DIR, 'df_fiat.csv'), index=False)

# 전체 wide format
df_merged = df.copy()
df_merged.to_csv(os.path.join(OUT_DIR, 'df_merged.csv'), index=False)

print(f'df_usdt : {df_usdt.shape}  |  depeg_USDT: {df_usdt["depeg_USDT"].sum()}일 ({df_usdt["depeg_USDT"].mean()*100:.1f}%)')
print(f'df_usdc : {df_usdc.shape}  |  depeg_USDC: {df_usdc["depeg_USDC"].sum()}일 ({df_usdc["depeg_USDC"].mean()*100:.1f}%)')
print(f'df_dai  : {df_dai.shape}   |  depeg_DAI : {df_dai["depeg_DAI"].sum()}일 ({df_dai["depeg_DAI"].mean()*100:.1f}%)')
print(f'df_fiat : {df_fiat.shape}')
print(f'df_merged: {df_merged.shape}')
print('저장 완료')

전부 NaN 컬럼 9개 제거: ['unreleased_USDT', 'minted_USDT', 'eth_minted_USDT', 'unreleased_USDC', 'minted_USDC', 'eth_minted_USDC', 'unreleased_DAI', 'minted_DAI', 'eth_minted_DAI']


df_usdt : (2287, 85)  |  depeg_USDT: 97일 (4.2%)
df_usdc : (2287, 83)  |  depeg_USDC: 28일 (1.2%)
df_dai  : (2287, 84)   |  depeg_DAI : 13일 (0.6%)
df_fiat : (4574, 83)
df_merged: (2316, 218)
저장 완료


---
## Step 6. 실제 데이터 확인

전처리 결과가 올바르게 생성되었는지 확인합니다.

In [7]:
# ── 6-1. 파일별 기본 정보 ────────────────────────────────────────────────────
print('=' * 55)
print('파일별 크기 및 기간')
print('=' * 55)

files = {
    'df_usdt': df_usdt,
    'df_usdc': df_usdc,
    'df_dai' : df_dai,
    'df_fiat': df_fiat,
}
for name, d in files.items():
    d['Date'] = pd.to_datetime(d['Date'])
    print(f'{name}: {d.shape[0]}행 × {d.shape[1]}열 | {d["Date"].min().date()} ~ {d["Date"].max().date()}')

파일별 크기 및 기간
df_usdt: 2287행 × 85열 | 2019-12-21 ~ 2026-03-25
df_usdc: 2287행 × 83열 | 2019-12-21 ~ 2026-03-25
df_dai: 2287행 × 84열 | 2019-12-21 ~ 2026-03-25
df_fiat: 4574행 × 83열 | 2019-12-21 ~ 2026-03-25


In [8]:
# ── 6-2. 타깃 변수(Y) 분포 ───────────────────────────────────────────────────
print('=' * 55)
print('타깃 변수 분포 (클래스 불균형 확인)')
print('=' * 55)

for name, d, col in [
    ('USDT', df_usdt, 'depeg_USDT'),
    ('USDC', df_usdc, 'depeg_USDC'),
    ('DAI',  df_dai,  'depeg_DAI'),
]:
    n0 = (d[col] == 0).sum()
    n1 = (d[col] == 1).sum()
    ratio = n0 / n1 if n1 > 0 else float('inf')
    print(f'{name}: 정상(0)={n0}일 | 디페깅(1)={n1}일 | 비율={ratio:.0f}:1 → SMOTE 필요')

타깃 변수 분포 (클래스 불균형 확인)
USDT: 정상(0)=2190일 | 디페깅(1)=97일 | 비율=23:1 → SMOTE 필요
USDC: 정상(0)=2259일 | 디페깅(1)=28일 | 비율=81:1 → SMOTE 필요
DAI: 정상(0)=2274일 | 디페깅(1)=13일 | 비율=175:1 → SMOTE 필요


In [9]:
# ── 6-3. 실제 데이터 샘플 (처음 5행) ─────────────────────────────────────────
print('=== USDT 데이터 샘플 (앞 5행) ===')
# 핵심 컬럼만 선택해서 확인
key_cols_usdt = [
    'Date', 'close_USDT', 'depeg_USDT', 'thresh_low_USDT', 'thresh_high_USDT',
    'return_1d_USDT', 'vol_7d_USDT', 'volume_ratio_USDT',
    'supply_change_USDT', 'vix', 'dxy', 'fgi'
]
df_usdt[key_cols_usdt].head()

=== USDT 데이터 샘플 (앞 5행) ===


,Date,close_USDT,depeg_USDT,thresh_low_USDT,thresh_high_USDT,return_1d_USDT,vol_7d_USDT,volume_ratio_USDT,supply_change_USDT,vix,dxy,fgi
0,2019-12-21,1.005758,1,0.99664,1.00336,-0.000083,0.006999,0.827195,13559170.0,12.51,97.709999,23.0
1,2019-12-22,1.003887,1,0.99664,1.00336,-0.001861,0.007015,0.967586,-6700000.0,12.51,97.709999,20.0
2,2019-12-23,1.012136,1,0.99664,1.00336,0.008218,0.007666,1.104159,0.0,12.61,97.660004,33.0
3,2019-12-24,1.008587,1,0.99664,1.00336,-0.003507,0.006172,0.964048,14108800.0,12.67,97.650002,25.0
4,2019-12-25,1.004661,1,0.99664,1.00336,-0.003892,0.004308,0.898116,3816650.0,12.67,97.650002,22.0


In [10]:
# ── 6-4. 디페깅 발생일 실제 데이터 확인 ─────────────────────────────────────
print('=== USDT 디페깅 이벤트 발생일 샘플 ===')
depeg_days = df_usdt[df_usdt['depeg_USDT'] == 1][
    ['Date', 'close_USDT', 'thresh_low_USDT', 'thresh_high_USDT',
     'volume_ratio_USDT', 'vix', 'btc_return_1d']
].head(10)
depeg_days

=== USDT 디페깅 이벤트 발생일 샘플 ===


,Date,close_USDT,thresh_low_USDT,thresh_high_USDT,volume_ratio_USDT,vix,btc_return_1d
0,2019-12-21,1.005758,0.996640,1.003360,0.827195,12.51,-0.003831
1,2019-12-22,1.003887,0.996640,1.003360,0.967586,12.51,0.044559
2,2019-12-23,1.012136,0.996640,1.003360,1.104159,12.61,-0.020763
3,2019-12-24,1.008587,0.996640,1.003360,0.964048,12.67,-0.004499
4,2019-12-25,1.004661,0.996640,1.003360,0.898116,12.67,-0.006470
5,2019-12-26,1.003988,0.996640,1.003360,1.008200,12.65,-0.004974
6,2019-12-27,1.005803,0.996640,1.003360,0.995563,13.43,0.007062
8,2019-12-29,1.004210,0.996640,1.003360,0.979014,13.43,0.014302
9,2019-12-30,1.006175,0.996640,1.003360,1.021602,14.82,-0.017468
13,2020-01-03,1.004192,0.996509,1.003491,1.247216,14.02,0.051452


In [11]:
print('=== DAI 디페깅 이벤트 발생일 전체 ===')
# DAI는 이벤트 수가 적으므로 전체 출력
df_dai[df_dai['depeg_DAI'] == 1][
    ['Date', 'close_DAI', 'thresh_low_DAI', 'thresh_high_DAI',
     'eth_return_1d', 'vol_7d_DAI', 'vix']
]

=== DAI 디페깅 이벤트 발생일 전체 ===


,Date,close_DAI,thresh_low_DAI,thresh_high_DAI,eth_return_1d,vol_7d_DAI,vix
60,2020-02-19,0.964845,0.974101,1.025899,-0.078670,0.024972,14.380000
83,2020-03-13,1.092951,0.964051,1.035949,0.185627,0.029287,57.830002
93,2020-03-23,1.036931,0.964051,1.035949,0.093986,0.038641,61.590000
107,2020-04-06,1.049618,0.964208,1.035792,0.178264,0.018896,45.240002
259,2020-09-05,1.029191,0.971189,1.028811,-0.136464,0.010424,30.750000
262,2020-09-08,1.031309,0.971189,1.028811,-0.042735,0.010214,31.459999
263,2020-09-09,1.031362,0.971189,1.028811,0.040011,0.009338,28.809999
264,2020-09-10,1.042337,0.971189,1.028811,0.048395,0.006220,29.709999
265,2020-09-11,1.043515,0.971189,1.028811,0.017913,0.006058,26.870001
266,2020-09-12,1.033794,0.971189,1.028811,0.033327,0.007473,26.870001


In [12]:
# ── 6-5. 주요 변수 기초 통계 ─────────────────────────────────────────────────
print('=== USDT 주요 변수 기초통계 ===')
stats_cols = [
    'close_USDT', 'return_1d_USDT', 'vol_7d_USDT',
    'volume_ratio_USDT', 'supply_change_USDT', 'mint_intensity_USDT',
    'vix', 'dxy', 'fgi', 'btc_return_1d'
]
df_usdt[stats_cols].describe().round(4)

=== USDT 주요 변수 기초통계 ===


,close_USDT,return_1d_USDT,vol_7d_USDT,volume_ratio_USDT,supply_change_USDT,mint_intensity_USDT,vix,dxy,fgi,btc_return_1d
count,2287.0000,2287.0000,2287.0000,2287.0000,2.287000e+03,2287.0000,2287.0000,2287.0000,2287.0000,2287.0000
mean,1.0003,-0.0000,0.0009,1.0054,7.915724e+07,0.0018,20.7968,100.0456,48.7617,0.0015
std,0.0020,0.0023,0.0023,0.2553,3.303085e+08,0.0112,7.6940,5.3777,22.5684,0.0318
min,0.9742,-0.0512,0.0000,0.3875,-3.205151e+09,-0.0674,11.8600,89.4400,5.0000,-0.3717
25%,0.9999,-0.0002,0.0002,0.8422,0.000000e+00,0.0000,15.7850,96.2850,28.0000,-0.0128
50%,1.0002,-0.0000,0.0003,0.9855,4.582560e+06,0.0001,18.9000,100.0300,50.0000,0.0003
75%,1.0005,0.0002,0.0006,1.1374,1.101020e+08,0.0014,23.8700,104.1650,70.0000,0.0151
max,1.0536,0.0548,0.0317,2.6041,3.000000e+09,0.3255,82.6900,114.1100,95.0000,0.1875


In [13]:
# ── 6-6. 결측치 잔여 확인 ────────────────────────────────────────────────────
print('=== 결측치 잔여 확인 ===')
for name, d in [('USDT', df_usdt), ('USDC', df_usdc), ('DAI', df_dai)]:
    miss = d.isnull().sum()
    miss_cols = miss[miss > 0]
    if len(miss_cols) == 0:
        print(f'{name}: 결측치 없음 ✅')
    else:
        print(f'{name}: 결측 있는 컬럼 {len(miss_cols)}개')
        print(miss_cols.to_string())

=== 결측치 잔여 확인 ===
USDT: 결측 있는 컬럼 1개
vol_30d_USDT    1
USDC: 결측 있는 컬럼 1개
vol_30d_USDC    1
DAI: 결측 있는 컬럼 12개
dai_eth_return_corr_30d     1
circ_DAI                    8
eth_circ_DAI                8
vol_30d_DAI                 1
turnover_rate_DAI          31
mcap_change_1d_DAI         31
mcap_change_7d_DAI         31
mint_intensity_DAI         31
supply_growth_rate_DAI      8
mcap_change_1d_DAI_lag1    32
mcap_change_1d_DAI_lag3    34
mcap_change_1d_DAI_lag7    38


In [14]:
# ── 6-7. 파생변수 그룹별 생성 확인 ───────────────────────────────────────────
print('=== USDT 파생변수 그룹별 목록 ===')
groups = {
    '가격 기반':     [c for c in df_usdt.columns if any(x in c for x in ['return_', 'vol_', 'rsi_', 'ma', 'shadow', 'spread_USDT', 'magnitude'])],
    '거래량 기반':   [c for c in df_usdt.columns if any(x in c for x in ['volume_ratio', 'volume_surge', 'turnover', 'vol_ma'])],
    '온체인 기반':   [c for c in df_usdt.columns if any(x in c for x in ['supply_change', 'mint_intensity', 'supply_growth', 'circ_', 'unreleased', 'tron', 'eth_circ'])],
    '거시경제 기반': [c for c in df_usdt.columns if any(x in c for x in ['dxy', 'vix', 'rate_hike', 'btc_dom', 'risk_off', 'btc_crash', 'btc_vol', 'eth_vol', 'crypto_stress', 'fgi', 'extreme'])],
    '라그 변수':     [c for c in df_usdt.columns if '_lag' in c],
}
for grp, cols in groups.items():
    print(f'\n[{grp}] ({len(cols)}개)')
    print(', '.join(cols))

=== USDT 파생변수 그룹별 목록 ===

[가격 기반] (32개)
btc_return_1d, eth_return_1d, btc_vol_7d, eth_vol_7d, vol_share_USDT, vol_share_USDC, vol_share_DAI, market_cap_USDT, return_1d_USDT, return_3d_USDT, return_7d_USDT, depeg_magnitude_USDT, high_low_spread_USDT, upper_shadow_USDT, lower_shadow_USDT, ma7_USDT, ma30_USDT, ma7_dev_USDT, ma30_dev_USDT, vol_7d_USDT, vol_30d_USDT, rsi_14_USDT, vol_ma7_USDT, return_1d_USDT_lag1, return_1d_USDT_lag3, return_1d_USDT_lag7, vol_7d_USDT_lag1, vol_7d_USDT_lag3, vol_7d_USDT_lag7, depeg_magnitude_USDT_lag1, depeg_magnitude_USDT_lag3, depeg_magnitude_USDT_lag7

[거래량 기반] (7개)
vol_ma7_USDT, volume_ratio_USDT, volume_surge_USDT, turnover_rate_USDT, volume_ratio_USDT_lag1, volume_ratio_USDT_lag3, volume_ratio_USDT_lag7

[온체인 기반] (11개)
circ_USDT, supply_change_USDT, eth_circ_USDT, eth_supply_change_USDT, tron_circ_USDT, tron_supply_change_USDT, mint_intensity_USDT, supply_growth_rate_USDT, supply_change_USDT_lag1, supply_change_USDT_lag3, supply_change_USDT_lag7

[거시경제

In [15]:
# ── 6-8. df_fiat long format 구조 확인 ──────────────────────────────────────
print('=== df_fiat (USDT+USDC 통합) 구조 확인 ===')
print(f'Shape: {df_fiat.shape}')
print(f'코인별 행 수:')
print(df_fiat['coin'].value_counts())
print(f'\n디페깅 이벤트:')
print(df_fiat.groupby('coin')['depeg_coin'].agg(['sum', 'mean']).round(3))
print(f'\n샘플 (USDT 5행):')
df_fiat[df_fiat['coin']=='USDT'][['Date','coin','close_coin','depeg_coin','vix','fgi']].head()

=== df_fiat (USDT+USDC 통합) 구조 확인 ===
Shape: (4574, 83)
코인별 행 수:
coin
USDC    2287
USDT    2287
Name: count, dtype: int64

디페깅 이벤트:
      sum   mean
coin            
USDC   28  0.012
USDT   97  0.042

샘플 (USDT 5행):


,Date,coin,close_coin,depeg_coin,vix,fgi
1,2019-12-21,USDT,1.005758,1,12.51,23.0
3,2019-12-22,USDT,1.003887,1,12.51,20.0
5,2019-12-23,USDT,1.012136,1,12.61,33.0
7,2019-12-24,USDT,1.008587,1,12.67,25.0
9,2019-12-25,USDT,1.004661,1,12.67,22.0
